In [6]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_chroma import Chroma
from langchain.schema import Document
from dotenv import load_dotenv


load_dotenv()

embedding = HuggingFaceEndpointEmbeddings(
    repo_id="sentence-transformers/all-MiniLM-L6-v2",
    task="feature-extraction"
)


In [7]:
# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [8]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [9]:
vector_store = Chroma(
    embedding_function=embedding,
    persist_directory='my_chroma_db',
    collection_name='sample'
)

In [10]:
# add documents
vector_store.add_documents(docs)

['87520903-d3ea-4a30-be81-fb715b7a3be2',
 '00654daa-1dbb-4e03-ae20-8146db86e8d0',
 'fbe12158-1bb5-48d1-8c2a-f887ece1900f',
 'f4c6f5bb-7005-4806-87c4-378bb44e193a',
 '3bf4f365-0fa0-4d70-ab36-d5fa061506dd']

In [11]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['87520903-d3ea-4a30-be81-fb715b7a3be2',
  '00654daa-1dbb-4e03-ae20-8146db86e8d0',
  'fbe12158-1bb5-48d1-8c2a-f887ece1900f',
  'f4c6f5bb-7005-4806-87c4-378bb44e193a',
  '3bf4f365-0fa0-4d70-ab36-d5fa061506dd'],
 'embeddings': array([[ 0.00994718,  0.06914335, -0.05147116, ..., -0.03543343,
          0.0128481 ,  0.01248287],
        [ 0.0012775 ,  0.03129857, -0.0237538 , ..., -0.0051836 ,
         -0.03280616,  0.02737708],
        [-0.10265914,  0.02650807,  0.02271501, ..., -0.03359747,
         -0.07984945, -0.01507713],
        [ 0.0212339 , -0.02468549, -0.0449436 , ..., -0.10995808,
          0.00572554,  0.09915374],
        [ 0.01873968,  0.04382843, -0.04304254, ..., -0.07801618,
         -0.07840687, -0.00304189]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [13]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=1
)

[Document(id='f4c6f5bb-7005-4806-87c4-378bb44e193a', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

In [14]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='f4c6f5bb-7005-4806-87c4-378bb44e193a', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9693602919578552),
 (Document(id='00654daa-1dbb-4e03-ae20-8146db86e8d0', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  1.1493452787399292)]

In [15]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="",
    filter={"team": "Chennai Super Kings"}
)

[(Document(id='fbe12158-1bb5-48d1-8c2a-f887ece1900f', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.8436005115509033),
 (Document(id='3bf4f365-0fa0-4d70-ab36-d5fa061506dd', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.8909374475479126)]

In [16]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='09a39dc6-3ba6-4ea7-927e-fdda591da5e4', document=updated_doc1)


In [17]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['87520903-d3ea-4a30-be81-fb715b7a3be2',
  '00654daa-1dbb-4e03-ae20-8146db86e8d0',
  'fbe12158-1bb5-48d1-8c2a-f887ece1900f',
  'f4c6f5bb-7005-4806-87c4-378bb44e193a',
  '3bf4f365-0fa0-4d70-ab36-d5fa061506dd'],
 'embeddings': array([[ 0.00994718,  0.06914335, -0.05147116, ..., -0.03543343,
          0.0128481 ,  0.01248287],
        [ 0.0012775 ,  0.03129857, -0.0237538 , ..., -0.0051836 ,
         -0.03280616,  0.02737708],
        [-0.10265914,  0.02650807,  0.02271501, ..., -0.03359747,
         -0.07984945, -0.01507713],
        [ 0.0212339 , -0.02468549, -0.0449436 , ..., -0.10995808,
          0.00572554,  0.09915374],
        [ 0.01873968,  0.04382843, -0.04304254, ..., -0.07801618,
         -0.07840687, -0.00304189]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [18]:
# delete document
vector_store.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])

In [19]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])


{'ids': ['87520903-d3ea-4a30-be81-fb715b7a3be2',
  '00654daa-1dbb-4e03-ae20-8146db86e8d0',
  'fbe12158-1bb5-48d1-8c2a-f887ece1900f',
  'f4c6f5bb-7005-4806-87c4-378bb44e193a',
  '3bf4f365-0fa0-4d70-ab36-d5fa061506dd'],
 'embeddings': array([[ 0.00994718,  0.06914335, -0.05147116, ..., -0.03543343,
          0.0128481 ,  0.01248287],
        [ 0.0012775 ,  0.03129857, -0.0237538 , ..., -0.0051836 ,
         -0.03280616,  0.02737708],
        [-0.10265914,  0.02650807,  0.02271501, ..., -0.03359747,
         -0.07984945, -0.01507713],
        [ 0.0212339 , -0.02468549, -0.0449436 , ..., -0.10995808,
          0.00572554,  0.09915374],
        [ 0.01873968,  0.04382843, -0.04304254, ..., -0.07801618,
         -0.07840687, -0.00304189]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo